# 03 – Train the Model

We have a sparse user-movie matrix. Now we factorise it.

**Alternating Least Squares (ALS)** decomposes the matrix into two low-rank factor matrices:
- **User factors** U: shape (n_users, k) — each user is a k-dimensional vector
- **Item factors** V: shape (n_movies, k) — each movie is a k-dimensional vector

Predicted rating = U[user] · V[movie] (dot product)

ALS alternates between solving for U (fixing V) and solving for V (fixing U),
repeating until convergence. Both sub-problems are simple linear regression.

The result: each user and movie has a compact representation (embedding) that
encodes taste/genre/style without ever using explicit labels.

In [ ]:
import sys
sys.path.insert(0, "../src")

from rec_sys.data import MovieLensLoader
from rec_sys.model import ALSTrainer

loader = MovieLensLoader("../data/ml-1m")
loader.load()
print(f"Loaded {loader.num_users:,} users, {loader.num_items:,} movies, {len(loader.ratings):,} ratings")

## Training

`ALSTrainer` wraps the `implicit` library. Key hyperparameters:
- `factors` (k): embedding dimensionality. More = more expressive, slower, more memory.
- `iterations`: how many ALS passes. 10–20 is usually enough for convergence.
- `reg`: L2 regularisation. Prevents overfitting by penalising large factor values.

In [ ]:
import time

trainer = ALSTrainer(factors=64, iterations=20, reg=0.01)

start = time.time()
trainer.train(
    ratings=loader.ratings,
    num_users=loader.num_users,
    num_items=loader.num_items,
)
elapsed = time.time() - start

print(f"Training time    : {elapsed:.1f}s")
print(f"User factor shape: {trainer.user_factors.shape}")
print(f"Item factor shape: {trainer.item_factors.shape}")

## Inspecting the Embeddings

Each user and movie now has a 64-dimensional vector. Let's look at what the
factors for a couple of users look like.

The values are continuous and don't have explicit meaning — the model learns
whatever representation minimises the reconstruction error on the observed ratings.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Sample three users and visualise their factor vectors.
fig, axes = plt.subplots(3, 1, figsize=(12, 5))
for i, uid in enumerate([0, 100, 500]):
    factors = trainer.user_factors[uid]
    axes[i].bar(range(len(factors)), factors, color="steelblue", alpha=0.7)
    axes[i].set_ylabel(f"User {uid}")
    axes[i].axhline(0, color="black", linewidth=0.5)
axes[-1].set_xlabel("Factor dimension")
fig.suptitle("User Factor Vectors (64 dimensions)")
plt.tight_layout()
plt.show()

## Dot Product as Similarity

The predicted affinity of user u for movie i is `U[u] · V[i]`.

Two vectors with high dot product are "aligned" — the user's taste vector
points in the same direction as the movie's feature vector.

Let's score user 1's affinity for a few known movies:

In [ ]:
from rec_sys.data.schema import UserId, MovieId

# User 1's history.
uid = loader.remapped_user_id(1)
user_ratings = [r for r in loader.ratings if r.user_id == uid]
print(f"User 1 has rated {len(user_ratings)} movies")

user_vec = trainer.get_user_vector(uid)

# Score a sample of movies and show the top and bottom.
sample_movie_ids = list(loader.movies.keys())[:200]
scores = []
for mid in sample_movie_ids:
    item_vec = trainer.get_item_vector(mid)
    score = user_vec.dot(item_vec)
    movie = loader.get_movie(mid)
    scores.append((movie.title, score))

scores.sort(key=lambda x: x[1], reverse=True)

print("\nTop 5 predicted:")
for title, score in scores[:5]:
    print(f"  {title:<50s} {score:.4f}")

print("\nBottom 5 predicted:")
for title, score in scores[-5:]:
    print(f"  {title:<50s} {score:.4f}")

## Factor Norms: What the Model Learned

The *norm* (length) of an item's factor vector measures how strongly opinionated
the model is about that item. High norm = divisive movie (people either love or hate it).
Low norm = neutral movie (everyone rates it similarly, nothing to distinguish).

In [ ]:
import numpy as np
from rec_sys.data.schema import MovieId

# Compute norms for all item vectors.
item_norms = []
for mid, movie in list(loader.movies.items())[:500]:  # First 500 for speed
    vec = trainer.get_item_vector(mid)
    norm = vec.norm()
    item_norms.append((movie.title, norm))

item_norms.sort(key=lambda x: x[1], reverse=True)

print("Most 'opinionated' movies (high norm = model has strong signal):")
for title, norm in item_norms[:8]:
    print(f"  {title:<50s} {norm:.4f}")

print("\nMost 'neutral' movies (low norm = weak signal / few ratings):")
for title, norm in item_norms[-8:]:
    print(f"  {title:<50s} {norm:.4f}")

## Key Takeaways

1. **Embeddings are learned** — ALS discovers latent factors from co-rating patterns,
   not from movie metadata. Genre, tone, era emerge implicitly.

2. **Dot product = predicted affinity** — higher score means the model thinks
   the user will like the item. This is what drives recommendations.

3. **Factor norms reveal item signal strength** — popular/divisive movies have
   bigger vectors because the model has more evidence for them.

4. **64 factors is a reasonable default** — too few and the model underfits;
   too many and it overfits and is slow. 32–128 is the typical range.

Next: `04_serve_recommendations.ipynb` — using these embeddings to generate real recs.